# Feature Engineering

## Objective

Load the raw Garmin export through the reusable cleaning pipeline, validate the cleaned dataset, and create transparent running features for later exploratory analysis.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cleaning import clean_garmin_activities

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "run_data.csv"

raw_activities = pd.read_csv(DATA_PATH)
runs = clean_garmin_activities(raw_activities)

print("Raw activities:", len(raw_activities))
print("Cleaned running activities:", len(runs))

runs.head()

Raw activities: 169
Cleaned running activities: 169


,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Avg Power,Max Power,Steps,Body Battery Drain,Best Lap Time,Number of Laps,Moving Time,Elapsed Time,Min Elevation,Max Elevation
0,Running,2026-07-11 09:15:11,False,Toronto Running,19.0,1225,0 days 01:47:07,166,185,4.7,...,259,433,19564,-23,0 days 00:00:01,20,0 days 01:46:49,0 days 01:57:28,111,182
1,Running,2026-07-09 07:40:06,False,Toronto Running,9.47,588,0 days 00:49:47,170,191,4.2,...,264,472,8744,-18,0 days 00:00:19.600000,22,0 days 00:48:59,0 days 01:01:14,75,81
2,Running,2026-07-08 17:36:39,False,Toronto Running,7.51,490,0 days 00:49:20,150,162,3.0,...,223,355,8816,-8,0 days 00:03:16,8,0 days 00:49:13,0 days 00:51:47,91,118
3,Treadmill Running,2026-07-07 09:13:03,False,Treadmill Running,2.61,79,0 days 00:08:11.100000,144,176,2.4,...,386,894,808,-4,0 days 00:08:11.100000,1,0 days 00:07:10,0 days 00:18:08,<NA>,<NA>
4,Treadmill Running,2026-07-06 07:40:51,False,Treadmill Running,4.27,424,0 days 00:46:41,144,155,2.6,...,130,303,4310,-9,0 days 00:00:09.500000,32,0 days 00:32:40,0 days 00:47:16,<NA>,<NA>


In [3]:
runs["Activity Type"].value_counts()
runs.dtypes
runs.isna().sum().sort_values(ascending=False).head(10)

Total Ascent               74
Min Elevation              72
Total Descent              71
Max Elevation              67
Body Battery Drain          3
Avg Vertical Ratio          1
Best Pace                   1
Max Power                   1
Avg Power                   1
Normalized Power® (NP®)     1
dtype: int64

In [4]:
from src.features import add_duration_features

featured_runs = add_duration_features(runs)

featured_runs[
    ["Moving Time", "moving_minutes", "Elapsed Time", "elapsed_minutes"]
].head()

,Moving Time,moving_minutes,Elapsed Time,elapsed_minutes
0,0 days 01:46:49,106.816667,0 days 01:57:28,117.466667
1,0 days 00:48:59,48.983333,0 days 01:01:14,61.233333
2,0 days 00:49:13,49.216667,0 days 00:51:47,51.783333
3,0 days 00:07:10,7.166667,0 days 00:18:08,18.133333
4,0 days 00:32:40,32.666667,0 days 00:47:16,47.266667


In [5]:
featured_runs[
    ["moving_minutes", "elapsed_minutes"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
moving_minutes,168.0,46.192589,23.525844,3.150,34.062500,47.000000,57.383333,109.033333
elapsed_minutes,169.0,60.219418,28.644304,4.405,41.283333,57.366667,82.233333,143.783333


In [6]:
from src.features import add_speed_features

featured_runs = add_speed_features(featured_runs)

featured_runs[
    ["Distance", "moving_minutes", "Avg Pace", "average_speed_kmh"]
].head()

,Distance,moving_minutes,Avg Pace,average_speed_kmh
0,19.0,106.816667,5.633333,10.672492
1,9.47,48.983333,5.250000,11.599864
2,7.51,49.216667,6.566667,9.155435
3,2.61,7.166667,3.133333,21.851163
4,4.27,32.666667,10.950000,7.842857
